In [25]:
import pandas as pd
df = pd.read_parquet("../scripts/fantom_inference/outputs_gen/train_gen.parquet")
df.head(10).values

array([['What does Veronica believe about how Fatima overcomes the challenge of dealing with different personalities at work?\n(a) Veronica believes that Fatima overcomes the challenge of dealing with different personalities at work by observing, learning, and adopting certain assertive communication strategies.\n(b) Veronica does not know or is unaware about how Fatima overcomes the challenge of dealing with different personalities at work because she was not involved in the conversation when this was discussed.\n\nChoose an answer from above:',
        'tom:belief:inaccessible:multiple-choice', 'first-order', '(b)',
        'Veronica believes that Fatima overcomes the challenge of dealing with different personalities at work by observing, learning, and adopting certain assertive communication strategies.',
        'inaccessible',
        "Christian: So, guys, how do you deal with challenges at your respective jobs?\nJaiden: I believe challenges are opportunities in disguise. They all

In [14]:
import re


def extract_answer(solution_str):
    # Extract answer between tags
    answer_pattern = r"<answer>(.*?)</answer>"
    match = re.finditer(answer_pattern, solution_str)
    matches = list(match)
    if matches:
        final_answer = matches[-1].group(1).strip().lower()
        final_answer = final_answer.rstrip(".")
    else:
        final_answer = None
    return final_answer

In [26]:
response_extracted = []
for idx, row in df.iterrows():
    answer = extract_answer(row['responses'][0])
    if answer is None:
        response_extracted.append("invalid")
    elif row['question_type'] in [
        "tom:belief:accessible:multiple-choice",
        "tom:belief:inaccessible:multiple-choice",
    ]:
        if answer in ["(a)", "(b)"]:
            response_extracted.append(answer)
        else:
            response_extracted.append("a-b-invalid")
    elif row['question_type'] in [
        "tom:answerability:binary",
        "tom:info_accessibility:binary",
    ]:
        if answer in ["yes", "no"]:
            response_extracted.append(answer)
        else:
            response_extracted.append("yes-no-invalid")
df['response_extracted'] = response_extracted

In [16]:
df['response_extracted'].value_counts()

response_extracted
yes        490
no         237
(b)         80
(a)         71
invalid     15
Name: count, dtype: int64

In [ ]:
df.loc[df['response_extracted'] == df['correct_answer']].shape[0]

507

In [22]:
df.loc[(df['response_extracted'].isin(['yes', 'no'])) & (df['correct_answer'].isin(['(a)', '(b)']))]

,question,question_type,tom_type,correct_answer,wrong_answer,missed_info_accessibility,context,input_text,set_id,choices_text,choices_list,data_source,prompt,ability,reward_model,extra_info,responses,response_extracted


In [ ]:
# test_gen
df[['response_extracted', 'correct_answer']].value_counts()

response_extracted  correct_answer
yes                 yes               342
                    no                148
no                  yes               144
                    no                 93
(b)                 (a)                41
                    (b)                39
(a)                 (b)                38
                    (a)                33
invalid             no                  7
                    yes                 4
                    (a)                 2
                    (b)                 2
Name: count, dtype: int64

In [20]:
df.shape[0]

893

In [27]:
# train_gen
df[["response_extracted", "correct_answer"]].value_counts()

response_extracted  correct_answer
yes                 yes               3075
                    no                1321
no                  yes               1112
                    no                 779
(b)                 (a)                362
(a)                 (a)                341
(b)                 (b)                335
(a)                 (b)                298
invalid             yes                 67
                    no                  49
                    (b)                 24
                    (a)                 22
a-b-invalid         (a)                  2
                    (b)                  1
yes-no-invalid      yes                  1
Name: count, dtype: int64